In [1]:
!pip install yt-dlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00


In [24]:
import shutil
from google.colab import files
shutil.make_archive('New_Playlist_Backup_CokeStudio_BanlgaMix', 'zip', '/content/music_download_CokeStudio')
#files.download('New_Playlist_Backup_CokeStudio_BanlgaMix.zip')


'/content/New_Playlist_Backup_CokeStudio_BanlgaMix.zip'

In [25]:
import os
import subprocess
import shutil
import sys

# ==========================================
# 1. ENVIRONMENT DETECTION
# ==========================================
IS_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content" if IS_COLAB else os.getcwd()
DOWNLOAD_DIR = os.path.join(BASE_DIR, "music_download")

print(f"Running in {'Google Colab' if IS_COLAB else 'Local Environment'}")

Running in Google Colab


In [26]:
# ==========================================
# 2. DEPENDENCY CHECK & INSTALL
# ==========================================
def setup_dependencies():
    print("Checking dependencies...")
    # Update yt-dlp
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "yt-dlp[default]"])

    if IS_COLAB:
        # Colab specific: Install Deno for signature solving
        if not shutil.which("deno"):
            print("Installing Deno (Colab)...")
            subprocess.run("curl -fsSL https://deno.land/install.sh | sh", shell=True)
            os.environ['PATH'] += ":/root/.deno/bin"
    else:
        # Local Machine: Check for essentials
        missing = [pkg for pkg in ["ffmpeg", "deno"] if not shutil.which(pkg)]
        if missing:
            print(f"WARNING: Missing local packages: {missing}")
            print("Please install them via: sudo apt install ffmpeg (for Ubuntu) or brew install (for Mac)")

setup_dependencies()

Checking dependencies...


In [27]:
# ==========================================
# 3. AUTHENTICATION (COOKIES)
# ==========================================
# Instructions for the user
print("\n--- AUTHENTICATION ---")
print("To download private playlists (like 'Your Likes'), paste your browser cookie string.")
print("If you don't need authentication (public playlists), just press Enter.")

raw_cookies = input("\nPaste 'cookie:' string (or leave blank): ").strip()
COOKIE_FILE = os.path.join(BASE_DIR, "youtube_cookies.txt")

if raw_cookies:
    if raw_cookies.startswith("cookie: "):
        raw_cookies = raw_cookies[8:]
    with open(COOKIE_FILE, "w") as f:
        f.write("# Netscape HTTP Cookie File\n")
        f.write(f".youtube.com\tTRUE\t/\tTRUE\t2147483647\tCOOKIE_DATA\t{raw_cookies}\n")
    print("Cookie file created.")
else:
    COOKIE_FILE = None
    print("Proceeding without cookies.")


--- AUTHENTICATION ---
To download private playlists (like 'Your Likes'), paste your browser cookie string.
If you don't need authentication (public playlists), just press Enter.

Paste 'cookie:' string (or leave blank): 
Proceeding without cookies.


In [33]:
import re

# ==========================================
# 4. DOWNLOAD ENGINE (SKIPS EXISTING & MINIMAL LOGS)
# ==========================================
playlist_url = input("\nPaste the YouTube Music Playlist URL: ").strip()

# Path for the archive file to keep track of downloads
ARCHIVE_FILE = os.path.join(BASE_DIR, "downloaded_history.txt")

if not os.path.exists(DOWNLOAD_DIR):
    os.makedirs(DOWNLOAD_DIR)

cmd = [
    "yt-dlp", "-x", "--audio-format", "mp3", "--audio-quality", "0",
    "--js-runtime", "deno",
    "--extractor-args", "youtube:player_client=ios,web",
    "--remote-components", "ejs:github",
    "--newline",
    "--download-archive", ARCHIVE_FILE,  # <--- This handles the skipping logic
    "-o", f"{DOWNLOAD_DIR}/%(title)s.%(ext)s",
    playlist_url
]

if COOKIE_FILE:
    cmd.extend(["--cookies", COOKIE_FILE])

print(f"\n🚀 Checking playlist... (Already downloaded songs will be skipped)\n")

with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    for line in process.stdout:
        # Check if yt-dlp is skipping the file
        if "has already been recorded in the archive" in line:
            # Extract name if possible, otherwise generic skip message
            print("⏭️ Skipping: Already exists in archive.")

        elif "[download] Destination:" in line:
            filename = line.split("Destination: ")[1].strip()
            print(f"📥 Downloading: {filename}")

        elif "[ExtractAudio]" in line:
            print("🎵 Converting to MP3...")

        elif "100% of" in line and "[download]" in line:
            print("✅ Done.")
            print("-" * 30)

if process.returncode == 0:
    print("\n✨ Sync complete!")


Paste the YouTube Music Playlist URL: https://music.youtube.com/playlist?list=PLVs9ErJcwUpvorU2yTWNkbMqUMEWISqcL

🚀 Checking playlist... (Already downloaded songs will be skipped)

🎵 Converting to MP3...
🎵 Converting to MP3...
🎵 Converting to MP3...
🎵 Converting to MP3...
🎵 Converting to MP3...
📥 Downloading: /content/music_download/Raataan Lambiyan – Official Video ｜ Shershaah ｜ Sidharth – Kiara ｜ Tanishk B｜ Jubin Nautiyal  ｜Asees.mp4
✅ Done.
------------------------------
🎵 Converting to MP3...
📥 Downloading: /content/music_download/ARZIYAAN.mp4
✅ Done.
------------------------------
🎵 Converting to MP3...
📥 Downloading: /content/music_download/Tera Hi Rahoon.mp4
✅ Done.
------------------------------
🎵 Converting to MP3...
📥 Downloading: /content/music_download/Sau Aasmaan.mp4
✅ Done.
------------------------------
🎵 Converting to MP3...
📥 Downloading: /content/music_download/Rang Lagala (feat. Anandi Joshi).mp4
✅ Done.
------------------------------
🎵 Converting to MP3...
📥 Downlo

In [ ]:

# ==========================================
# 5. EXPORT
# ==========================================
if IS_COLAB:
    from google.colab import files
    print("\nZipping and triggering browser download...")
    shutil.make_archive('Music_Export', 'zip', DOWNLOAD_DIR)
    files.download('Music_Export.zip')
else:
    print(f"\nSuccess! Your music is located in: {DOWNLOAD_DIR}")